In [ ]:
import cv2 
import os

# Buscamos dinámicamente la primera imagen que exista en la carpeta 'aranas'
# Esto evita errores si los nombres cambiaron 
aranas = os.path.join('dataset_limpio', 'Aranas')

if os.path.exists(aranas):
    nombres_archivos = os.listdir(aranas)
    
    if len(nombres_archivos) > 0:
        # Tomamos el primer archivo que encuentre
        primera_imagen = nombres_archivos[0] 
        ruta_completa = os.path.join(aranas, primera_imagen)

        img = cv2.imread(ruta_completa)

        if img is not None:
            print(f"✅ Éxito leyendo imagen: {primera_imagen}")
            print("Dimensiones detectadas:", img.shape) # Debería ser (64, 64, 3)
        else:
            print("❌ Error: El archivo existe pero cv2 no pudo leerlo.")
    else:
        print("❌ La carpeta 'Aranas' está vacía.")
else:
    print("❌ No encuentro la carpeta dataset/aranas.")

# Convolutional Neural Networks

# Importar Librerías

In [ ]:
import numpy as np
import os
import re
import matplotlib.pyplot as plt
# %matplotlib inline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

print("Todo instalado correctamente")

In [ ]:
import tensorflow as tf

from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential, Model, load_model
from tensorflow.keras.layers import (
    Input,
    Dense,
    Dropout,
    Flatten,
    Conv2D,
    MaxPooling2D,
    BatchNormalization,
    SeparableConv2D,
    Activation,
    LeakyReLU
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Cargar set de Imágenes

In [ ]:
from skimage.transform import resize
dirname = os.path.join(os.getcwd(),'dataset_limpio')
imgpath = dirname + os.sep 

images = []
directories = []
dircount = []
prevRoot=''
cant=0

print("leyendo imagenes de ",imgpath)

for root, dirnames, filenames in os.walk(imgpath):
    for filename in filenames:
        if re.search("\.(jpg|jpeg|png|bmp|tiff)$", filename):
            cant=cant+1
            filepath = os.path.join(root, filename)
            image = plt.imread(filepath)
             #  Convertir en RGB si es en escala de grises
            if len(image.shape) == 2:
                image = np.stack((image,) * 3, axis=-1)
            images.append(image.astype(np.uint8))
            b = "Leyendo..." + str(cant)
            print (b, end="\r")
            if prevRoot !=root:
                print(root, cant)
                prevRoot=root
                directories.append(root)
                dircount.append(cant)
                cant=0
dircount.append(cant)

dircount = dircount[1:]
dircount[0]=dircount[0]+1
print('Directorios leidos:',len(directories))
print("Imagenes en cada directorio", dircount)
print('suma Total de imagenes en subdirs:',sum(dircount))

# Creamos las etiquetas

In [ ]:
labels=[]
indice=0
for cantidad in dircount:
    for i in range(cantidad):
        labels.append(indice)
    indice=indice+1
print("Cantidad etiquetas creadas: ",len(labels))


In [ ]:
deportes=[]
indice=0
for directorio in directories:
    name = directorio.split(os.sep)
    print(indice , name[len(name)-1])
    deportes.append(name[len(name)-1])
    indice=indice+1

In [ ]:
y = np.array(labels)
X = np.array(images, dtype=np.uint8) #convierto de lista a numpy



# Find the unique numbers from the train labels
classes = np.unique(y)
nClasses = len(classes)
print('Total number of outputs : ', nClasses)
print('Output classes : ', classes)

# Creamos Sets de Entrenamiento y Test

In [ ]:
train_X,test_X,train_Y,test_Y = train_test_split(X,y,test_size=0.2, random_state=42, stratify=y)
print('Training data shape : ', train_X.shape, train_Y.shape)
print('Testing data shape : ', test_X.shape, test_Y.shape)

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_X = train_X.astype('float32') / 255.0

test_X = test_X.astype('float32') / 255.0

train_Y_one_hot = to_categorical(train_Y)
test_Y_one_hot = to_categorical(test_Y)

train_X, valid_X, train_label, valid_label = train_test_split(
    train_X,
    train_Y_one_hot,
    test_size=0.2,
    random_state=13,
    stratify=train_Y
)

# Configuración del Data Augmentation
# Esto creará variaciones infinitas de tus fotos en tiempo real
datagen = ImageDataGenerator(
    rotation_range=8,
    width_shift_range=0.04,
    height_shift_range=0.04,
    zoom_range=0.05,
    horizontal_flip=True,
    fill_mode='constant',
    cval=128/255.0
)

# Calculamos las estadísticas necesarias usando tus datos de entrenamiento
datagen.fit(train_X)
print("✅ Generador de imágenes configurado.")

In [ ]:
plt.figure(figsize=[5,5])

# Display the first image in training data
plt.subplot(121)
plt.imshow(train_X[0,:,:], cmap='gray')
plt.title("Ground Truth : {}".format(train_Y[0]))

# Display the first image in testing data
plt.subplot(122)
plt.imshow(test_X[0,:,:], cmap='gray')
plt.title("Ground Truth : {}".format(test_Y[0]))

# Preprocesamos las imagenes

In [ ]:
#train_X = train_X.astype('float32')
#test_X = test_X.astype('float32')
#train_X = train_X/255.
#test_X = test_X/255.
plt.imshow(test_X[0,:,:])

## Hacemos el One-hot Encoding para la red

In [ ]:
# Change the labels from categorical to one-hot encoding
#train_Y_one_hot = to_categorical(train_Y)
#test_Y_one_hot = to_categorical(test_Y)

# Display the change for category label using one-hot encoding
print('Original label:', train_Y[0])
print('After conversion to one-hot:', train_Y_one_hot[0])

# Creamos el Set de Entrenamiento y Validación

In [ ]:
#Mezclar todo y crear los grupos de entrenamiento y testing
#train_X,valid_X,train_label,valid_label = train_test_split(train_X, train_Y_one_hot, test_size=0.2, random_state=13, stratify=train_Y)

In [ ]:
print(train_X.shape,valid_X.shape,train_label.shape,valid_label.shape)

# Creamos el modelo de CNN

In [ ]:
#declaramos variables con los parámetros de configuración de la red
INIT_LR = 1e-3 # Valor inicial de learning rate. El valor 1e-3 corresponde con 0.001
epochs = 20 # Cantidad de iteraciones completas al conjunto de imagenes de entrenamiento
batch_size = 32 # cantidad de imágenes que se toman a la vez en memoria

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, Input

sport_model = Sequential([
    Input(shape=(32, 32, 3)),
    
    # Bloque 1
    Conv2D(32, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D(2,2),
    

    # Bloque 2
    Conv2D(64, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D(2,2),
     

    # Bloque 3
    Conv2D(128, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D(2,2),
    
    

    Flatten(),
    Dense(128, activation='relu'),
    Dense(nClasses, activation='softmax')
])

# Compilamos con una tasa de aprendizaje (learning rate) un poco más baja para ser precisos
from tensorflow.keras.optimizers import Adam
sport_model.compile(optimizer=Adam(learning_rate=0.0005), 
                    loss='categorical_crossentropy', 
                    metrics=['accuracy'])

sport_model.summary()

In [ ]:
sport_model.summary()

In [ ]:
from tensorflow.keras.optimizers import Adam

# Bajamos la velocidad inicial a 0.0005 para ser más cautelosos al principio
sport_model.compile(optimizer=Adam(learning_rate=0.0005), 
                    loss='categorical_crossentropy', 
                    metrics=['accuracy'])

print("Modelo compilado con Learning Rate inicial de 0.0005")

# Entrenamos el modelo: Aprende a clasificar imágenes

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# 1. Early Stopping: Si no mejora en 8 épocas, para y recupera lo mejor.
early_stop = EarlyStopping(
    monitor='val_loss', 
    patience=8, 
    restore_best_weights=True,
    verbose=1
)

# 2. Reduce Learning Rate: Si se estanca 3 veces, baja la velocidad a la mitad.
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss', 
    factor=0.5,       
    patience=3,       
    min_lr=0.00001,   
    verbose=1         
)

print("Iniciando entrenamiento (Con Data Augmentation + ReduceLR)...")

# 3. Entrenar
sport_train = sport_model.fit(
    datagen.flow(train_X, train_label, batch_size=32),
    epochs=20, 
    verbose=1,
    validation_data=(valid_X, valid_label),
    callbacks=[early_stop, reduce_lr] # <--- ¡Ambos activados!
)

print("Entrenamiento finalizado.")

In [ ]:
# guardamos la red, para reutilizarla en el futuro, sin tener que volver a entrenar
sport_model.save("Modelos/animales.keras")

# Evaluamos la red

In [ ]:
test_eval = sport_model.evaluate(test_X, test_Y_one_hot, verbose=1)

In [ ]:
print('Test loss:', test_eval[0])
print('Test accuracy:', test_eval[1])

In [ ]:
sport_train.history

In [ ]:
accuracy = sport_train.history['accuracy']
val_accuracy = sport_train.history['val_accuracy']
loss = sport_train.history['loss']
val_loss = sport_train.history['val_loss']
epochs = range(len(accuracy))
plt.plot(epochs, accuracy, 'bo', label='Training accuracy')
plt.plot(epochs, val_accuracy, 'b', label='Validation accuracy')
plt.title('Training and validation accuracy')
plt.legend()
plt.figure()
plt.plot(epochs, loss, 'bo', label='Training loss')
plt.plot(epochs, val_loss, 'b', label='Validation loss')
plt.title('Training and validation loss')
plt.legend()
plt.show()

In [ ]:
predicted_classes2 = sport_model.predict(test_X)

In [ ]:
predicted_classes=[]
for predicted_sport in predicted_classes2:
    predicted_classes.append(predicted_sport.tolist().index(max(predicted_sport)))
predicted_classes=np.array(predicted_classes)

In [ ]:
predicted_classes.shape, test_Y.shape

# Aprendamos de los errores: Qué mejorar

In [ ]:
correct = np.where(predicted_classes==test_Y)[0]
print("Found %d correct labels" % len(correct))
for i, correct in enumerate(correct[0:9]):
    plt.subplot(3,3,i+1)

    # --- AQUÍ ESTÁ EL CAMBIO: (64, 64, 3) ---
    plt.imshow(test_X[correct].reshape(32,32,3), cmap='gray', interpolation='none')

    plt.title("{}, {}".format(deportes[predicted_classes[correct]],
                                                    deportes[test_Y[correct]]))

    plt.tight_layout()

In [ ]:
incorrect = np.where(predicted_classes!=test_Y)[0]
print("Found %d incorrect labels" % len(incorrect))

for i, incorrect in enumerate(incorrect[0:9]):
    plt.subplot(3,3,i+1)

    # --- CORRECCIÓN AQUÍ: Cambiar a (64,64,3) ---
    plt.imshow(test_X[incorrect].reshape(32,32,3), cmap='gray', interpolation='none')

    plt.title("{}, {}".format(deportes[predicted_classes[incorrect]],
                                                    deportes[test_Y[incorrect]]))
    plt.tight_layout()

In [ ]:
target_names = ["Class {}".format(i) for i in range(nClasses)]
print(classification_report(test_Y, predicted_classes, target_names=target_names))

In [41]:
import os
import re
import numpy as np
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import load_img, img_to_array

# Carpeta donde están las imágenes de prueba
test_dir = 'tests'

# Modelo entrenado
modelo_h5 = 'Modelos/animales.keras'

# Clases del modelo
sriesgos = ['Aranas', 'ballenas', 'changos', 'pajaros', 'ranas']

# Extensiones válidas
image_extensions = re.compile(r"\.(jpg|jpeg|png|bmp|tiff|webp)$", re.IGNORECASE)

# Verificar que exista el modelo
if not os.path.exists(modelo_h5):
    raise FileNotFoundError(f"No existe el modelo: {modelo_h5}")

# Cargar modelo
riesgo_model = load_model(modelo_h5)
print("Modelo cargado exitosamente.")

images = []
filenames_found = []

# Verificar que exista la carpeta tests
if not os.path.exists(test_dir):
    raise FileNotFoundError(f"No existe la carpeta de pruebas: {test_dir}")

print(f"\nBuscando imágenes en: {test_dir}")

for filename in os.listdir(test_dir):
    if image_extensions.search(filename):
        filepath = os.path.join(test_dir, filename)

        try:
            # Carga la imagen como RGB y la redimensiona a 32x32
            image = load_img(filepath, target_size=(32, 32), color_mode='rgb')
            image_array = img_to_array(image)

            images.append(image_array)
            filenames_found.append(filename)

        except Exception as e:
            print(f"Error al leer/procesar {filename}. Error: {e}")

if not images:
    print("\nNo se encontraron imágenes válidas o se falló al leer todas.")
else:
    X = np.array(images, dtype='float32')
    test_X = X/255

    predicted_classes = riesgo_model.predict(test_X)
    predicted_labels = np.argmax(predicted_classes, axis=1)

    print("\n--- Resultados ---")

for i, label_index in enumerate(predicted_labels):
    print(f"\nImagen: {filenames_found[i]}")
    print(f"Predicción final: {sriesgos[label_index]}")
    print("Porcentajes:")

    for j, probabilidad in enumerate(predicted_classes[i]):
        porcentaje = probabilidad * 100
        print(f"  {sriesgos[j]}: {porcentaje:.2f}%")

Modelo cargado exitosamente.

Buscando imágenes en: tests
1/1 [==============================] - 0s 254ms/step

--- Resultados ---

Imagen: mono.jpg
Predicción final: Aranas
Porcentajes:
  Aranas: 94.80%
  ballenas: 0.00%
  changos: 0.34%
  pajaros: 0.01%
  ranas: 4.85%

Imagen: mono2.png
Predicción final: changos
Porcentajes:
  Aranas: 8.10%
  ballenas: 0.00%
  changos: 91.83%
  pajaros: 0.04%
  ranas: 0.04%

Imagen: mono3.png
Predicción final: changos
Porcentajes:
  Aranas: 0.09%
  ballenas: 0.00%
  changos: 99.91%
  pajaros: 0.00%
  ranas: 0.00%

Imagen: mono4.png
Predicción final: Aranas
Porcentajes:
  Aranas: 95.62%
  ballenas: 0.00%
  changos: 0.02%
  pajaros: 0.00%
  ranas: 4.36%

Imagen: mono5.png
Predicción final: ranas
Porcentajes:
  Aranas: 29.19%
  ballenas: 0.00%
  changos: 0.08%
  pajaros: 0.00%
  ranas: 70.73%
